# Day 61：毕业设计开发日 3 —— 多工具协同、持久化记忆与 HITL 审批

> **版本**：`v1.0.0-day61` | **对应 Issue**：GitLab Issue #061
> **实操目标**：验证 `CapstoneAgentOrchestrator` 整合财务比率计算器、企业 RAG 检索、PII 合规脱敏、SQLite 多轮记忆与高管报告发布前 HITL 人工审批中断/恢复全链路。

## 1. 环境初始化与导入 `project_iterations/day_61` 模块

In [1]:
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'project_iterations').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DAY_DIR = REPO_ROOT / 'project_iterations' / 'day_61'
if str(DAY_DIR) not in sys.path:
    sys.path.insert(0, str(DAY_DIR))

from zhishu_agent import (
    ApprovalStatus,
    CapstoneAgentOrchestrator,
    ComplianceGuard,
    FinancialMetricsInput,
    FinancialRatioTool,
    RAGSearchTool,
)

print('Day 61 模块导入成功:', DAY_DIR)

Day 61 模块导入成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_61


## 2. 单工具能力验证：财务比率计算器、RAG 检索与 PII 脱敏网关

In [2]:
fin_tool = FinancialRatioTool()
rag_tool = RAGSearchTool()
guard = ComplianceGuard()

metrics = FinancialMetricsInput(
    company_name='星辰制造科技',
    period='2026Q3',
    revenue=12000.0,
    net_profit=480.0,
    total_assets=25000.0,
    total_liabilities=19000.0,
    shareholders_equity=6000.0,
    current_assets=5200.0,
    current_liabilities=4800.0,
    gross_profit=3600.0,
)
fin_res = fin_tool.calculate(metrics)
print('财务指标:', fin_res.to_dict())

rag_hits = rag_tool.search('高杠杆 资产负债率 流动比率', top_k=2)
for hit in rag_hits:
    print(f'RAG 命中 [{hit.doc_id}] {hit.title} (score={hit.score})')

raw_text = '联系人张总 13800138000，邮箱 risk@xingchen.ai，密钥 sk-prod9988776655。'
sanitized_text, counts = guard.redact(raw_text)
print('脱敏结果:', sanitized_text)
print('脱敏统计:', counts)

财务指标: {'company_name': '星辰制造科技', 'period': '2026Q3', 'roe_pct': 8.0, 'net_profit_margin_pct': 4.0, 'gross_margin_pct': 30.0, 'debt_to_asset_ratio_pct': 76.0, 'current_ratio': 1.08, 'asset_turnover': 0.48, 'risk_level': 'HIGH', 'risk_flags': ['高杠杆预警：资产负债率 76.0% >= 70%', '短期偿债压力：流动比率 1.08 < 1.2']}
RAG 命中 [DOC-RISK-001] 企业高杠杆与流动性风控指引（2026修订版） (score=0.4302)
RAG 命中 [DOC-FIN-002] 高管经营分析与 ROE 盈利质量评估标准 (score=0.1404)
脱敏结果: 联系人张总 [REDACTED_PHONE]，邮箱 [REDACTED_EMAIL]，密钥 [REDACTED_API_KEY]。
脱敏统计: {'api_key': 1, 'phone': 1, 'email': 1}


## 3. 多轮对话持久化记忆与 HITL 审批中断/恢复端到端演练

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / 'capstone_day61.db'
    agent = CapstoneAgentOrchestrator(db_path=db_path)
    session_id = 'nb-exec-session'

    # Turn 1: 常规多轮问询（自动脱敏后写入 SQLite）
    t1 = agent.run(
        session_id=session_id,
        user_query='请查询高管报告 HITL 审批规范，联系手机 13912345678。',
    )
    print('[Turn 1]', t1.status.value, '| redactions:', t1.redaction_counts)
    print('  回复:', t1.reply)

    # Turn 2: 触发高管风险报告发布 -> 自动进入 HITL 中断状态
    t2 = agent.run(
        session_id=session_id,
        user_query='请基于 2026Q3 财报生成并发布高管风险评估报告，抄送 cfo@xingchen.ai。',
        financial_metrics=metrics,
        require_report_publish=True,
    )
    print('\n[Turn 2]', t2.status.value, '| approval_id:', t2.approval_id)
    assert t2.status == ApprovalStatus.PENDING_APPROVAL
    assert agent.memory.list_published_reports(session_id) == []

    # 模拟服务重启：使用全新 Orchestrator 实例连接同一 SQLite 完成审批放行
    resumed_agent = CapstoneAgentOrchestrator(db_path=db_path)
    t3 = resumed_agent.resume(
        approval_id=t2.approval_id,
        approved=True,
        reviewer='风控总监-李明',
        comment='财务指标与风控依据核验无误，同意正式发布。',
    )
    print('\n[Turn 3 - Resume]', t3.status.value)
    print('  回复:', t3.reply)
    print('  已发布报告标题:', t3.published_report.title)
    print('  SQLite 持久化消息数:', len(resumed_agent.memory.get_history(session_id)))

[Turn 1] completed | redactions: {'phone': 1}
  回复: 已结合 0 条历史上下文完成分析。 参考企业知识库：[DOC-HITL-004]高管风险报告发布人工审批（HITL）SOP；[DOC-COMP-003]智能体输出合规与 PII 个人敏感信息脱敏规范。

[Turn 2] pending_approval | approval_id: hitl-86284f7fea

[Turn 3 - Resume] approved
  回复: [审批通过] 复核人 风控总监-李明 已批准发布《星辰制造科技（2026Q3）高管风险评估报告》（report_id=rep-947cc984af）。
  已发布报告标题: 星辰制造科技（2026Q3）高管风险评估报告
  SQLite 持久化消息数: 5


## 4. 验证人工驳回分支（HITL Reject）与防重保护

In [4]:
with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / 'capstone_reject.db'
    agent = CapstoneAgentOrchestrator(db_path=db_path)
    pause = agent.run(
        session_id='nb-reject-session',
        user_query='申请发布季度综合风控报告。',
        require_report_publish=True,
    )
    rej = agent.resume(
        approval_id=pause.approval_id,
        approved=False,
        reviewer='法务合规-赵敏',
        comment='需补充关联交易明细后再提交。',
    )
    print('[Reject]', rej.status.value, '|', rej.reply)
    assert rej.published_report is None
    assert agent.memory.list_published_reports('nb-reject-session') == []

[Reject] rejected | [审批驳回] 复核人 法务合规-赵敏 拒绝发布《企业综合风控与合规高管评估报告》，原因：需补充关联交易明细后再提交。。
